# 04 — Evaluation

- `quick`: DIAGNOSTIC — seeded athlete subsample, smaller candidate sets; never compare with standard/full.
- `standard`: the 12F evaluation row (within / cross-domain retrieval, matched retrieval, domain AUC / MMD, physical probes).
- `full`: standard + shuffled-identity null.

Transductive and inductive results are separate tables; the NFL-transfer ladder uses inductive embeddings only.

In [ ]:
%pip install -q "git+https://github.com/YOUR_USERNAME/YOUR_REPO.git"
from google.colab import drive
drive.mount("/content/drive")
from nflbowl import Experiment

exp = Experiment.open("/content/drive/MyDrive/NFLBowl/experiments/f4_main", runtime={"stage_dir": "/content/nflbowl_stage"})
exp.prepare()

In [ ]:
# quick look while iterating (diagnostic only)
exp.evaluate("F4", level="quick")

In [ ]:
# research tables: baselines + every trained family at the full level
for name in ["F0", "random", "F2", "F3", "F4", "F5", "F6"]:
    try:
        exp.evaluate(name, level="full")
    except FileNotFoundError as e:
        print(name, "—", e)
exp.compare(level="full")

In [ ]:
# learning-curve runs (if trained in 02)
for frac in exp.cfg.science.learning_curve_fractions:
    try:
        exp.evaluate("F4", level="standard", nfl_fraction=frac)
    except FileNotFoundError as e:
        print(frac, "—", e)

In [ ]:
# INDUCTIVE held-out retrieval (separate table), with the engineered baseline per fold
import polars as pl

ind = pl.concat([exp.evaluate(n, regime="inductive") for n in ["F0", *exp.cfg.science.inductive_configs]], how="diagonal_relaxed")
ind.group_by("representation").agg(pl.col("auc").mean().alias("mean_auc"), pl.col("auc").min().alias("min_fold_auc"))

In [ ]:
# NFL transfer ladder on the 12C frozen target (needs athletes.parquet with nfl.q90_shared in the bundle)
exp.transfer()

In [ ]:
exp.plot()
print(exp.report())